# 설명 가능한 전통 ML 탐색

같은 학습 8 / 검증 1 / 평가 1 문서 LODO에서 **검증 선택 Logistic, NB-SVM, fastText**를 비교합니다. 평가 문서는 파라미터 선택에 쓰지 않습니다. 막대 평균만 보지 않고 fold별 변화, 혼동행렬, 근거 표현과 오류 사례를 함께 봅니다.

먼저 `python -m scripts.evaluation.classical_search`를 실행해 Git 제외된 `data/processed/` 결과를 만드세요.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager
from sklearn.metrics import confusion_matrix

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False
from scripts.evaluation.baselines import CHAR_BALANCED, LABELS, run_lodo, summarize
from scripts.labeling.label_dataset import load_label_dataset
rows, _ = load_label_dataset()
result_dir = ROOT / 'data/processed'
predictions = pd.read_csv(result_dir / 'classical_search_predictions.csv', encoding='utf-8-sig')
summary = json.loads((result_dir / 'classical_search_summary.json').read_text(encoding='utf-8'))
baseline = run_lodo(rows, CHAR_BALANCED)
print(f'평가 예측 {len(predictions):,}행 / 모델 {predictions.model.nunique()}개 / 기준선 macro F1 {summarize(baseline)["macro_f1"]["fold_mean"]:.3f}')

In [ ]:
baseline_by_fold = {r.test_document: r.macro_f1 for r in baseline}
fold_rows = []
for model, payload in summary.items():
    for fold in payload['folds']:
        fold_rows.append({'모델': model, '평가 문서': fold['test_document'], 'macro F1': fold['macro_f1'], '기준선 대비': fold['macro_f1'] - baseline_by_fold[fold['test_document']]})
fold_scores = pd.DataFrame(fold_rows)
score_table = pd.DataFrame({model: {metric: values['fold_mean'] for metric, values in payload['metrics'].items()} for model, payload in summary.items()}).T
display(score_table[['macro_f1', 'accuracy', 'review_precision', 'review_recall', 'review_f1']].style.format('{:.3f}').highlight_max(axis=0, color='#b7e4c7'))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.stripplot(data=fold_scores, x='모델', y='macro F1', hue='평가 문서', jitter=False, ax=axes[0])
axes[0].legend([], [], frameon=False); axes[0].set_title('문서별 macro F1')
sns.pointplot(data=fold_scores, x='모델', y='기준선 대비', errorbar=None, join=False, color='#264653', ax=axes[1])
sns.stripplot(data=fold_scores, x='모델', y='기준선 대비', color='#e76f51', ax=axes[1])
axes[1].axhline(0, color='black', linewidth=1); axes[1].set_title('고정 char Logistic 대비 fold 차이')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, predictions.model.nunique(), figsize=(15, 4))
for ax, (model, frame) in zip(axes, predictions.groupby('model', sort=False)):
    matrix = confusion_matrix(frame.gold, frame.pred, labels=LABELS, normalize='true')
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(model); ax.set_xlabel('예측'); ax.set_ylabel('정답')
plt.tight_layout(); plt.show()

In [ ]:
selected = []
for model, payload in summary.items():
    selected.extend({'모델': model, '선택 파라미터': value} for value in payload['selected_parameters'].values())
display(pd.DataFrame(selected).value_counts().rename('fold 수').reset_index())

feature_rows = []
for row in predictions.dropna(subset=['top_features']).itertuples():
    for item in str(row.top_features).split(' | '):
        if ':' not in item: continue
        feature, score = item.rsplit(':', 1)
        feature_rows.append({'모델': row.model, '예측 클래스': row.pred, '표현': feature, '기여도': float(score)})
features = pd.DataFrame(feature_rows)
top = (features.groupby(['모델', '예측 클래스', '표현'], as_index=False)['기여도'].sum().sort_values('기여도', ascending=False).groupby(['모델', '예측 클래스']).head(8))
display(top.sort_values(['모델', '예측 클래스', '기여도'], ascending=[True, True, False]))

In [ ]:
is_correct = predictions.correct.astype(str).str.lower().eq('true')
errors = predictions[~is_correct].copy()
errors = errors.sort_values(['model', 'score_margin'], ascending=[True, False])
display(errors[['model', 'test_document', 'requirement_uid', 'gold', 'pred', 'score_margin', 'top_features', 'text']].groupby('model').head(10))
print('점수가 높은 오류는 모델이 확신하며 틀린 사례다. feature 조각은 원문의 같은 부분과 함께 읽고, fastText는 직접 feature 계수를 제공하지 않아 빈칸으로 둔다.')